# Manipulations des règles

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle,re,features,pyperclip, operator
import networkx as nx
from scipy.stats import entropy
debug=False

In [3]:
%store -r ordStemCells
ordStemCells

['pi1S',
 'pi2S',
 'pi3S',
 'pi1P',
 'pi2P',
 'pi3P',
 'ii1S',
 'ii2S',
 'ii3S',
 'ii1P',
 'ii2P',
 'ii3P',
 'fi1S',
 'fi2S',
 'fi3S',
 'fi1P',
 'fi2P',
 'fi3P',
 'pc1S',
 'pc2S',
 'pc3S',
 'pc1P',
 'pc2P',
 'pc3P',
 'ps1S',
 'ps2S',
 'ps3S',
 'ps1P',
 'ps2P',
 'ps3P',
 'ai1S',
 'ai2S',
 'ai3S',
 'ai1P',
 'ai2P',
 'ai3P',
 'is1S',
 'is2S',
 'is3S',
 'is1P',
 'is2P',
 'is3P',
 'pI2S',
 'pI1P',
 'pI2P',
 'inf',
 'pP',
 'ppMS',
 'ppMP',
 'ppFS',
 'ppFP']

## Préparatifs

### Transformation du fichier des traits des phonèmes en tableau LaTeX

In [4]:
nFeatures='/Users/gilles/Github/SWIM/ParadigmGeneration/L4L/bdlexique.ini'
dfFeatures = pd.read_csv(nFeatures, sep="|", skiprows=4,index_col=0)
dfFeatures

,+son,-son,+syl,-syl,+cons,-cons,+ant,-ant,+cor,-cor,...,-nas,+lat,-lat,+cont,-cont,+voice,-voice,+strid,-strid,Unnamed: 31
,,,,,,,,,,,,,,,,,,,,,
p,,X,,X,X,,X,,,X,...,X,,X,,X,,X,,X,NaN
t,,X,,X,X,,X,,X,,...,X,,X,,X,,X,,X,NaN
k,,X,,X,X,,,X,,X,...,X,,X,,X,,X,,X,NaN
b,,X,,X,X,,X,,,X,...,X,,X,,X,X,,,X,NaN
d,,X,,X,X,,X,,X,,...,X,,X,,X,X,,,X,NaN
g,,X,,X,X,,,X,,X,...,X,,X,,X,X,,,X,NaN
f,,X,,X,X,,X,,,X,...,X,,X,X,,,X,X,,NaN
s,,X,,X,X,,X,,X,,...,X,,X,X,,,X,X,,NaN
S,,X,,X,X,,,X,X,,...,X,,X,X,,,X,X,,NaN


In [5]:
df_initial=dfFeatures
print("--- DataFrame initial ---")
print(df_initial)

# 2. Liste de vos 16 traits (sans le prefixe '+' ou '-')
# Remplacez cette liste par les vrais noms de vos 16 traits
liste_traits = ["son", "syl", "cons", "ant","cor", "back", "high", "low", "round","ATR", "nas", "lat", "cont", "voice", "strid",]

# 3. Transformation
df_final = pd.DataFrame(index=df_initial.index)

for trait in liste_traits:
    col_plus = f"+{trait}"
    col_moins = f"-{trait}"

    # Conditions : vérifie où se trouvent les 1 (ou True)
    conditions = [
        df_initial[col_plus].str.strip() == "X",
        df_initial[col_moins].str.strip() == "X",
    ]
    choix = ["+", "-"]

    # np.select attribue '+' si col_plus==1, '-' si col_moins==1, sinon None
    df_final[trait] = np.select(conditions, choix, default=None)

print("\n--- DataFrame final ---")
print(df_final.to_latex())

--- DataFrame initial ---
      +son  -son  +syl  -syl  +cons  -cons  +ant  -ant  +cor  -cor  ...  -nas  \
                                                                    ...         
p            X           X      X            X                 X    ...   X     
t            X           X      X            X           X          ...   X     
k            X           X      X                  X           X    ...   X     
b            X           X      X            X                 X    ...   X     
d            X           X      X            X           X          ...   X     
g            X           X      X                  X           X    ...   X     
f            X           X      X            X                 X    ...   X     
s            X           X      X            X           X          ...   X     
S            X           X      X                  X     X          ...   X     
v            X           X      X            X                 X    ...   X     
z 

### Préparation du codage des phonèmes

In [6]:
features.add_config(nFeatures)
fs=features.FeatureSystem('phonemes')
# fs.context.objects


In [7]:
# traduire SAMPA-BDLex en API

def sampa2api(sampa):
    api=sampa
    api=api.replace(u'n"',u'n') 
    api=api.replace(u't"',u't') 
    api=api.replace(u'z"',u'z') 
    api=api.replace(u'R"',u'ʁ') 
    api=api.replace(u'p"',u'p') 
    api=api.replace(u'S',u'ʃ') 
    api=api.replace(u'Z',u'ʒ')
    api=api.replace(u'N',u'ŋ')
    api=api.replace(u'J',u'ɲ')
    # api=api.replace(u'r',u'ʁ') 
    api=api.replace(u'H',u'ɥ')
    api=api.replace(u'E',u'ɛ')
    api=api.replace(u'2',u'ø')
    api=api.replace(u'9',u'œ')
    api=api.replace(u'6',u'ə')
    api=api.replace(u'O',u'ɔ')
    api=api.replace(u'è',u'e')   
    api=api.replace(u'ò',u'o')    
    api=api.replace(u'â',u'ɑ̃')   
    api=api.replace(u'ê',u'ɛ̃')   
    api=api.replace(u'û',u'œ̃')  
    api=api.replace(u'ô',u'ɔ̃')       
    api=api.replace(u'@',u'ə')
    api=api.replace(u'R',u'ʁ') 
    return api

### Préparation des structures pour les règles

In [8]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n, file=logfile)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

    
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=True):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug:
                    print (forme, file=logfile)
                    print ("pas de classe",idClasseForme, file=logfile)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)), file=logfile)
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme, file=logfile) 
                print ("pas de patron", file=logfile)
        return sortieForme
        

## Lecture du fichier de règles

In [9]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
rulesFile="vlexique2-Total-Regles.pkl"
fRules=repFiles+rulesFile
with open(fRules, 'rb') as input:
    regles = pickle.load(input)
# resultatsLecture[('fi1P', 'ai1P')].classeCF 

pFreq={}
for k in regles:
    for p in regles[k].patrons:
        if p not in pFreq:
            pFreq[p]=0
        pFreq[p]+=1

In [9]:
regles["inf","pP"].patrons

{'E-â': '^(.*[ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwHE96Oêûô])E$',
 'r-sâ': '^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])ir$',
 'r-â': '^(.*[mnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZjrwHiyEe926uOo])r$',
 'udr-Olvâ': '^([bdvzrE6a][pbjiEe][sz])udr$',
 'ir-â': '^(.*[ptkbdgfsSvzZmnJNjrwHiyEe926auOoêûâô][fvjrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjr])ir$',
 'r-jâ': '^(.*[ptbdfsvzr][jrwH][iEea])r$',
 'tr-sâ': '^(.*[ptkbdgfsSvzZmnJNjrwH][Ea])tr$',
 'êdr-aJâ': '^(.*)wêdr$',
 'HE-â': '^(.*)yHE$',
 'war-â': '^(.*[ptbdfsvzmnl][iyEe926auOo][vzlr])war$',
 'âdr-9nâ': '^(.*)prâdr$',
 'E-yâ': '^argE$',
 'war-Ejâ': '^(.*)swar$',
 'êdr-EJâ': '^(.*[ptkbdgfsSvzZlr])êdr$',
 'avwar-Ejâ': '^avwar$',
 'war-yvâ': '^(.*)bwar$',
 'r-zâ': '^(.*[ptkbdgfsSvzZjlrwH][iyEe926uOo])r$',
 'r-vâ': '^(.*[fsvzE])krir$',
 'wE-â': '^(.*[ptkbdgfsSvzZ][lr])uwE$',
 'Er-9zâ': '^(.*)fEr$',
 'dr-zâ': '^(.*)kudr$',
 '9..r-O..sâ': '^(.*)fl9rir$',
 'E.ir-i.â': '^ZEzir$',
 'dr-lâ': '^(.*)mudr$',
 'ir-jâ': '^(.*)rir$',
 'vwar-Sâ':

In [10]:
nRegles={
"E-â":"a",
"r-sâ":"b",
"r-â":"c",
"ir-â":"d",
"r-jâ":"e",
"tr-sâ":"f",
"war-â":"g",
"âdr-9nâ":"h",
"êdr-EJâ":"i",
"r-zâ":"j",
"r-vâ":"k",
"êdr-aJâ":"l",
"udr-Olvâ":"m",
"Er-9zâ":"n",
"ir-jâ":"o",
"dr-zâ":"p",
"war-Ejâ":"q",
"avwar-Ejâ":"r",
"war-yvâ":"s",
"E.ir-i.â":"t",
"dr-lâ":"u",
"vwar-Sâ":"v",
"war-Eâ":"w",
"9..r-O..sâ":"b1",
"E-yâ":"a2",
"HE-â":"a3",
"wE-â":"a4",
}

## tableau des classes de compétition avec contexte phonologique

In [40]:
cc=set()
parentheses=False
for k in regles[("inf","pP")].classe:
    if parentheses:
        cc.add("\{"+", ".join([nRegles[r] for r in k.split(", ") if len(nRegles[r])==1])+"\}")
    else:
        cc.add(", ".join([nRegles[r] for r in k.split(", ") if len(nRegles[r])==1]))
for c in sorted(cc):
    print(r"\{%s\} & xxx & xxx \\"%c)

\{a\} & xxx & xxx \\
\{b, c\} & xxx & xxx \\
\{b, c, d\} & xxx & xxx \\
\{b, c, d, e, j, o\} & xxx & xxx \\
\{b, c, d, j\} & xxx & xxx \\
\{b, c, d, j, o\} & xxx & xxx \\
\{b, c, e, j\} & xxx & xxx \\
\{b, c, e, j, o\} & xxx & xxx \\
\{b, c, j\} & xxx & xxx \\
\{b, c, j, k, o\} & xxx & xxx \\
\{b, c, j, o\} & xxx & xxx \\
\{b, d, j\} & xxx & xxx \\
\{b, d, j, t\} & xxx & xxx \\
\{b, j\} & xxx & xxx \\
\{c\} & xxx & xxx \\
\{c, e, j\} & xxx & xxx \\
\{c, f\} & xxx & xxx \\
\{c, h\} & xxx & xxx \\
\{c, i\} & xxx & xxx \\
\{c, j\} & xxx & xxx \\
\{c, l\} & xxx & xxx \\
\{c, m\} & xxx & xxx \\
\{c, p\} & xxx & xxx \\
\{c, u\} & xxx & xxx \\
\{e\} & xxx & xxx \\
\{e, g\} & xxx & xxx \\
\{e, g, v\} & xxx & xxx \\
\{e, q\} & xxx & xxx \\
\{e, q, w\} & xxx & xxx \\
\{e, r\} & xxx & xxx \\
\{e, s\} & xxx & xxx \\
\{g\} & xxx & xxx \\
\{j\} & xxx & xxx \\
\{j, n\} & xxx & xxx \\


In [41]:
classes={}
for c,cf in regles[("inf","pP")].classe.items():
    cClasse=", ".join(sorted([nRegles[r] for r in c.split(", ") if len(nRegles[r])==1]))
    # print(cClasse)
    if cClasse not in classes:
        classes[cClasse]={}
    # else:
        # print("màj classe",cClasse)
    for cfRegle,cfValue in cf.items():
        # print(classes[cClasse])
        if cfRegle not in classes[cClasse]:
            classes[cClasse][cfRegle]=0
        # else:
            # print('màj règle')
            # print(cfRegle,cfValue)
        classes[cClasse][cfRegle]+=cfValue
    # print("result",c,classes[cClasse])
    # print()

classes=dict(sorted(classes.items()))
populations={}
for k,v in classes.items():
    print(k,end=" : ")
    populations[k]={}
    for kk,vv in v.items():
        print(nRegles[kk],vv,end=", ")
        populations[k][nRegles[kk]]=vv
    print()

a : a 4546, a3 8, a2 1, a4 10, 
b, c : b 16, 
b, c, d : b 41, d 31, 
b, c, d, e, j, o : b 1, d 9, 
b, c, d, j : d 10, b 7, 
b, c, d, j, o : d 14, b 20, o 1, 
b, c, e, j : j 18, b 4, e 2, 
b, c, e, j, o : b 6, 
b, c, j : b 38, j 10, 
b, c, j, k, o : k 11, j 1, 
b, c, j, o : b 11, o 1, b1 2, 
b, d, j : b 165, j 10, d 18, 
b, d, j, t : t 1, 
b, j : b 37, j 1, 
c : c 61, 
c, e, j : e 7, 
c, f : c 24, f 19, 
c, h : h 11, 
c, i : i 20, 
c, j : j 7, c 4, 
c, l : l 8, 
c, m : m 3, 
c, p : p 3, 
c, u : u 3, 
e : e 7, 
e, g : g 14, 
e, g, v : v 1, 
e, q : q 3, e 3, 
e, q, w : q 1, w 1, 
e, r : r 1, 
e, s : s 2, 
g : g 4, 
j : j 1, 
j, n : n 8, 


In [42]:
print(len(populations))
for c,v in populations.items():
    print(c,v)
print()
print()
tPopulation=0
for c,v in populations.items():
    cPopulation=sum(v.values())
    tPopulation+=cPopulation
    cEntropie=entropy(list(v.values()),base=2)
    print(f"\\{{{c}\\}} & {cPopulation} & {cEntropie:.3f} \\\\")

print()
print(tPopulation)

34
a {'a': 4546, 'a3': 8, 'a2': 1, 'a4': 10}
b, c {'b': 16}
b, c, d {'b': 41, 'd': 31}
b, c, d, e, j, o {'b': 1, 'd': 9}
b, c, d, j {'d': 10, 'b': 7}
b, c, d, j, o {'d': 14, 'b': 20, 'o': 1}
b, c, e, j {'j': 18, 'b': 4, 'e': 2}
b, c, e, j, o {'b': 6}
b, c, j {'b': 38, 'j': 10}
b, c, j, k, o {'k': 11, 'j': 1}
b, c, j, o {'b': 11, 'o': 1, 'b1': 2}
b, d, j {'b': 165, 'j': 10, 'd': 18}
b, d, j, t {'t': 1}
b, j {'b': 37, 'j': 1}
c {'c': 61}
c, e, j {'e': 7}
c, f {'c': 24, 'f': 19}
c, h {'h': 11}
c, i {'i': 20}
c, j {'j': 7, 'c': 4}
c, l {'l': 8}
c, m {'m': 3}
c, p {'p': 3}
c, u {'u': 3}
e {'e': 7}
e, g {'g': 14}
e, g, v {'v': 1}
e, q {'q': 3, 'e': 3}
e, q, w {'q': 1, 'w': 1}
e, r {'r': 1}
e, s {'s': 2}
g {'g': 4}
j {'j': 1}
j, n {'n': 8}


\{a\} & 4565 & 0.044 \\
\{b, c\} & 16 & 0.000 \\
\{b, c, d\} & 72 & 0.986 \\
\{b, c, d, e, j, o\} & 10 & 0.469 \\
\{b, c, d, j\} & 17 & 0.977 \\
\{b, c, d, j, o\} & 35 & 1.137 \\
\{b, c, e, j\} & 24 & 1.041 \\
\{b, c, e, j, o\} & 6 & 0.000 \\
\{b, c, j\} 

In [45]:
print(len(populations))
for c,v in populations.items():
    print(c,v)
print()
print()
tPopulation=0
for c,v in populations.items():
    cPopulation=sum(v.values())
    tPopulation+=cPopulation
    cDistribution=", ".join([f"{r} : {p/cPopulation:.1%}" for r,p in dict(sorted(v.items())).items()])
    cDistribution=", ".join([f"{r} : {p}" for r,p in dict(sorted(v.items())).items()])
    print(f"\\{{{c}\\}} & {cPopulation} & {cDistribution} \\\\")

print()
print(tPopulation)

34
a {'a': 4546, 'a3': 8, 'a2': 1, 'a4': 10}
b, c {'b': 16}
b, c, d {'b': 41, 'd': 31}
b, c, d, e, j, o {'b': 1, 'd': 9}
b, c, d, j {'d': 10, 'b': 7}
b, c, d, j, o {'d': 14, 'b': 20, 'o': 1}
b, c, e, j {'j': 18, 'b': 4, 'e': 2}
b, c, e, j, o {'b': 6}
b, c, j {'b': 38, 'j': 10}
b, c, j, k, o {'k': 11, 'j': 1}
b, c, j, o {'b': 11, 'o': 1, 'b1': 2}
b, d, j {'b': 165, 'j': 10, 'd': 18}
b, d, j, t {'t': 1}
b, j {'b': 37, 'j': 1}
c {'c': 61}
c, e, j {'e': 7}
c, f {'c': 24, 'f': 19}
c, h {'h': 11}
c, i {'i': 20}
c, j {'j': 7, 'c': 4}
c, l {'l': 8}
c, m {'m': 3}
c, p {'p': 3}
c, u {'u': 3}
e {'e': 7}
e, g {'g': 14}
e, g, v {'v': 1}
e, q {'q': 3, 'e': 3}
e, q, w {'q': 1, 'w': 1}
e, r {'r': 1}
e, s {'s': 2}
g {'g': 4}
j {'j': 1}
j, n {'n': 8}


\{a\} & 4565 & a : 4546, a2 : 1, a3 : 8, a4 : 10 \\
\{b, c\} & 16 & b : 16 \\
\{b, c, d\} & 72 & b : 41, d : 31 \\
\{b, c, d, e, j, o\} & 10 & b : 1, d : 9 \\
\{b, c, d, j\} & 17 & b : 7, d : 10 \\
\{b, c, d, j, o\} & 35 & b : 20, d : 14, o : 1 \\
\{b, c,

## Tableau des classes des compétition sans contexte phonologique

In [46]:
cc=set()
parentheses=False
for k in regles[("inf","pP")].classeCF:
    if parentheses:
        cc.add("\{"+", ".join([nRegles[r] for r in k.split(", ") if len(nRegles[r])==1])+"\}")
    else:
        cc.add(", ".join([nRegles[r] for r in k.split(", ") if len(nRegles[r])==1]))
for c in sorted(cc):
    print(r"\{%s\} & xxx & xxx \\"%c)

\{a\} & xxx & xxx \\
\{b, c, d, e, j, k, o\} & xxx & xxx \\
\{b, c, d, e, j, k, t, o\} & xxx & xxx \\
\{b, c, e, f, j, k\} & xxx & xxx \\
\{b, c, e, g, q, r, s, j, k, v, w\} & xxx & xxx \\
\{b, c, e, g, q, s, j, k, v, w\} & xxx & xxx \\
\{b, c, e, g, q, s, j, k, w\} & xxx & xxx \\
\{b, c, e, h, j, k, p, u\} & xxx & xxx \\
\{b, c, e, j, k\} & xxx & xxx \\
\{b, c, e, j, k, n\} & xxx & xxx \\
\{b, c, e, j, k, p, u\} & xxx & xxx \\
\{b, c, e, l, i, j, k, p, u\} & xxx & xxx \\
\{b, c, m, e, j, k, p, u\} & xxx & xxx \\


In [47]:
classes={}
for c,cf in regles[("inf","pP")].classeCF.items():
    cClasse=", ".join(sorted([nRegles[r] for r in c.split(", ") if len(nRegles[r])==1]))
    # print(cClasse)
    if cClasse not in classes:
        classes[cClasse]={}
    # else:
        # print("màj classe",cClasse)
    for cfRegle,cfValue in cf.items():
        # print(classes[cClasse])
        if cfRegle not in classes[cClasse]:
            classes[cClasse][cfRegle]=0
        # else:
            # print('màj règle')
            # print(cfRegle,cfValue)
        classes[cClasse][cfRegle]+=cfValue
    # print("result",c,classes[cClasse])
    # print()

classes=dict(sorted(classes.items()))
populations={}
for k,v in classes.items():
    print(k,end=" : ")
    populations[k]={}
    for kk,vv in v.items():
        print(nRegles[kk],vv,end=", ")
        populations[k][nRegles[kk]]=vv
    print()

a : a 4546, a2 1, a3 8, a4 10, 
b, c, d, e, j, k, o : b 295, d 74, j 35, k 11, e 2, o 2, b1 2, 
b, c, d, e, j, k, o, t : b 51, d 8, j 5, t 1, 
b, c, e, f, j, k : c 27, f 19, 
b, c, e, g, j, k, q, r, s, v, w : r 1, v 1, 
b, c, e, g, j, k, q, s, v, w : g 13, e 6, 
b, c, e, g, j, k, q, s, w : q 4, e 4, s 2, g 5, w 1, 
b, c, e, h, j, k, p, u : c 28, h 11, 
b, c, e, i, j, k, l, p, u : l 8, i 20, 
b, c, e, j, k : j 4, c 15, 
b, c, e, j, k, m, p, u : m 3, p 3, u 3, 
b, c, e, j, k, n : e 7, j 4, n 8, 
b, c, e, j, k, p, u : c 19, 


In [48]:
print(len(populations))
for c,v in populations.items():
    print(c,v)
print()
print()
tPopulation=0
for c,v in populations.items():
    cPopulation=sum(v.values())
    tPopulation+=cPopulation
    cDistribution=", ".join([f"{r} : {p/cPopulation:.1%}" for r,p in v.items()])
    cDistribution=", ".join([f"{r} : {p}" for r,p in dict(sorted(v.items())).items()])
    print(f"\\{{{c}\\}} & {cPopulation} & {cDistribution} \\\\")

print()
print(tPopulation)

13
a {'a': 4546, 'a2': 1, 'a3': 8, 'a4': 10}
b, c, d, e, j, k, o {'b': 295, 'd': 74, 'j': 35, 'k': 11, 'e': 2, 'o': 2, 'b1': 2}
b, c, d, e, j, k, o, t {'b': 51, 'd': 8, 'j': 5, 't': 1}
b, c, e, f, j, k {'c': 27, 'f': 19}
b, c, e, g, j, k, q, r, s, v, w {'r': 1, 'v': 1}
b, c, e, g, j, k, q, s, v, w {'g': 13, 'e': 6}
b, c, e, g, j, k, q, s, w {'q': 4, 'e': 4, 's': 2, 'g': 5, 'w': 1}
b, c, e, h, j, k, p, u {'c': 28, 'h': 11}
b, c, e, i, j, k, l, p, u {'l': 8, 'i': 20}
b, c, e, j, k {'j': 4, 'c': 15}
b, c, e, j, k, m, p, u {'m': 3, 'p': 3, 'u': 3}
b, c, e, j, k, n {'e': 7, 'j': 4, 'n': 8}
b, c, e, j, k, p, u {'c': 19}


\{a\} & 4565 & a : 4546, a2 : 1, a3 : 8, a4 : 10 \\
\{b, c, d, e, j, k, o\} & 421 & b : 295, b1 : 2, d : 74, e : 2, j : 35, k : 11, o : 2 \\
\{b, c, d, e, j, k, o, t\} & 65 & b : 51, d : 8, j : 5, t : 1 \\
\{b, c, e, f, j, k\} & 46 & c : 27, f : 19 \\
\{b, c, e, g, j, k, q, r, s, v, w\} & 2 & r : 1, v : 1 \\
\{b, c, e, g, j, k, q, s, v, w\} & 19 & e : 6, g : 13 \\
\{b, c, e

## Liste des transformations par fréquence

In [18]:
dict(sorted(pFreq.items(), key=lambda item: item[1],reverse=True))

{'-': 597,
 'y-E': 220,
 'E-y': 220,
 'i-jE': 216,
 'jE-i': 216,
 '-jE': 209,
 'jE-': 209,
 'E-': 176,
 '-E': 176,
 '-HE': 156,
 'HE-': 156,
 'y-': 153,
 '-y': 153,
 '-s': 128,
 's-': 128,
 'i-E': 120,
 'E-i': 120,
 '9.E-E.': 120,
 'E.-9.E': 120,
 'E-9rE': 108,
 '9rE-E': 108,
 'HE-y': 108,
 'wE-u': 108,
 'wE-': 108,
 'y-HE': 108,
 'u-wE': 108,
 '-wE': 108,
 '-rE': 102,
 'rE-': 102,
 '-t': 100,
 't-': 100,
 '-z': 100,
 'z-': 100,
 'j.-r.': 98,
 'r.-j.': 98,
 'yHE-E': 96,
 'E-yHE': 96,
 'i-': 93,
 '-i': 93,
 '-jô': 87,
 'jô-': 87,
 'jE-irE': 84,
 'wajE-ErE': 84,
 'irE-jE': 84,
 'ErE-wajE': 84,
 'E-ô': 84,
 'ô-E': 84,
 '-zE': 84,
 'zE-': 84,
 'H.-r.': 82,
 'al.-ir.': 82,
 'r.-H.': 82,
 'ir.-al.': 82,
 'j-': 81,
 '-j': 81,
 'E-rE': 78,
 'w.-r.': 78,
 'rE-E': 78,
 'r.-w.': 78,
 'E-is': 78,
 'is-E': 78,
 'HE-yrE': 72,
 '9.E-E.9rE': 72,
 'wE-urE': 72,
 'E-y9rE': 72,
 'yrE-HE': 72,
 'E.9rE-9.E': 72,
 'urE-wE': 72,
 'y9rE-E': 72,
 '-sE': 72,
 '-vE': 72,
 'sE-': 72,
 'vE-': 72,
 'y-wa': 72,
 'yH

### Affichage des règles au format LaTeX

In [19]:
def stripC(text):
    result=text
    joker=r"\^\(\.\*"
    m=re.match(joker+r"(\w+)?(\[[^)]*)\)",text)
    mm=re.match(joker+r"(\w+)",text)
    mmm=re.match(r"\^(\w+)",text)
    if m:
        result="".join([g for g in m.groups() if g is not None])
    elif mm:
        result=mm.group(1)
    elif mmm:
        result=mmm.group(1)
    return sampa2api(result)
    
for k,v in regles[("inf","pP")].patrons.items():
    print(k,v)
    a,b = k.split("-")
    c=v.rsplit(a,1)
    print("\\ex \\phonc{\\textipan{%s}}{\\textipan{%s}}{\\textipan{%s} \\phold{} \\#}"%(sampa2api(a),sampa2api(b),stripC(c[0])))
    # print()

E-â ^(.*[ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwHE96Oêûô])E$
\ex \phonc{\textipan{ɛ}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəɔɛ̃œ̃ɔ̃]} \phold{} \#}
r-sâ ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])ir$
\ex \phonc{\textipan{r}}{\textipan{sɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəaɔɛ̃œ̃ɑ̃ɔ̃]} \phold{} \#}
r-â ^(.*[mnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZjrwHiyEe926uOo])r$
\ex \phonc{\textipan{r}}{\textipan{ɑ̃}}{\textipan{[mnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒjrwɥiyɛeœøəuɔo]} \phold{} \#}
udr-Olvâ ^([bdvzrE6a][pbjiEe][sz])udr$
\ex \phonc{\textipan{udr}}{\textipan{ɔlvɑ̃}}{\textipan{^([bdvzrɛəa][pbjiɛe][sz])} \phold{} \#}
ir-â ^(.*[ptkbdgfsSvzZmnJNjrwHiyEe926auOoêûâô][fvjrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjr])ir$
\ex \phonc{\textipan{ir}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][fvjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjr]} \phold{} \#}
r-jâ ^(.*[ptbdfsvzr][jrwH][iEea])r$
\ex \phonc{\textipan{r}}{

In [20]:
def stripC(text):
    result=text
    joker=r"\^\(\.\*"
    m=re.match(joker+r"(\w+)?(\[[^)]*)\)",text)
    mm=re.match(joker+r"(\w+)",text)
    mmm=re.match(r"\^(\w+)",text)
    if m:
        result="".join([g for g in m.groups() if g is not None])
    elif mm:
        result=mm.group(1)
    elif mmm:
        result=mmm.group(1)
    return sampa2api(result)
    
for k in nRegles:
    v=regles[("inf","pP")].patrons[k]
    # print(k,v)
    a,b = k.split("-")
    c=v.rsplit(a,1)
    print("%s \\ex \\phonc{\\textipan{%s}}{\\textipan{%s}}{\\textipan{%s} \\phold{} \\#}"%(nRegles[k],sampa2api(a),sampa2api(b),stripC(c[0])))
    # print()

a \ex \phonc{\textipan{ɛ}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəɔɛ̃œ̃ɔ̃]} \phold{} \#}
b \ex \phonc{\textipan{r}}{\textipan{sɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəaɔɛ̃œ̃ɑ̃ɔ̃]} \phold{} \#}
c \ex \phonc{\textipan{r}}{\textipan{ɑ̃}}{\textipan{[mnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒjrwɥiyɛeœøəuɔo]} \phold{} \#}
d \ex \phonc{\textipan{ir}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][fvjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjr]} \phold{} \#}
e \ex \phonc{\textipan{r}}{\textipan{jɑ̃}}{\textipan{[ptbdfsvzr][jrwɥ][iɛea]} \phold{} \#}
f \ex \phonc{\textipan{tr}}{\textipan{sɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjrwɥ][ɛa]} \phold{} \#}
g \ex \phonc{\textipan{war}}{\textipan{ɑ̃}}{\textipan{[ptbdfsvzmnl][iyɛeœøəauɔo][vzlr]} \phold{} \#}
h \ex \phonc{\textipan{ɑ̃dr}}{\textipan{œnɑ̃}}{\textipan{^(.*)pr} \phold{} \#}
i \ex \phonc{\textipan{ɛ̃dr}}{\textipan{ɛɲɑ̃}}{\textipan{[ptkbdgfsʃvzʒlr]} \phold{} \#}
j \ex \phonc{\texti

In [36]:
def stripC(text):
    result=text
    joker=r"\^\(\.\*"
    m=re.match(joker+r"(\w+)?(\[[^)]*)\)",text)
    mm=re.match(joker+r"(\w+)",text)
    mmm=re.match(r"\^(\w+)",text)
    if m:
        result="".join([g for g in m.groups() if g is not None])
    elif mm:
        result=mm.group(1)
    elif mmm:
        result=mmm.group(1)
    return sampa2api(result)
    
for k in nRegles:
    v=regles[("inf","pP")].patrons[k]
    # print(k,v)
    a,b = k.split("-")
    c=v.rsplit(a,1)
    print("\\ex \\phonc{\\textipan{%s}}{\\textipan{%s}}{\\phold{} \\#}"%(sampa2api(a),sampa2api(b)))
    # print()

\ex \phonc{\textipan{ɛ}}{\textipan{ɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{r}}{\textipan{sɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{r}}{\textipan{ɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ir}}{\textipan{ɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{r}}{\textipan{jɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{tr}}{\textipan{sɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{war}}{\textipan{ɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ɑ̃dr}}{\textipan{œnɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ɛ̃dr}}{\textipan{ɛɲɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{r}}{\textipan{zɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{r}}{\textipan{vɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ɛ̃dr}}{\textipan{aɲɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{udr}}{\textipan{ɔlvɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ɛr}}{\textipan{œzɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{ir}}{\textipan{jɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{dr}}{\textipan{zɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{war}}{\textipan{ɛjɑ̃}}{\phold{} \#}
\ex \phonc{\textipan{avwar}}{\textipan{ɛjɑ̃}}{\phold{} \#}


## Génération des formes candidates

In [ ]:
paires=regles.keys()
cases=set(k for k,v in paires)
case1="pi3S"
case2="fi3S"
forme1="brwa"
forme2="brwara"
c1={}
c2={}
common={}
vals={}
for case in cases:
    cVals=vals[case]={}
    c1[case]=regles[(case1,case)].sortirForme(forme1,contextFree=False)
    c2[case]=regles[(case2,case)].sortirForme(forme2,contextFree=False)
    print(case1,case,c1[case])
    print(case2,case,c2[case])

    common[case]=c1[case].keys() & c2[case].keys()
    
    for k in common[case]:
        cVals[k]=c1[case][k]+c2[case][k]
    tVals=sum(cVals.values())
    print()
    for k,v in cVals.items():
        cVals[k]=cVals[k]/tVals
        print(k, f"{cVals[k]:.2f}")    
    print()
    print("========================")



In [ ]:
vals

In [ ]:
print (case1, forme1, case2, forme2)
print()
for k1,v1 in vals.items():
    print(k1,end=" : ")
    for k2,v2 in dict(sorted(v1.items(), key=lambda x: x[1], reverse=True)).items():
        print(f"{v2:.0%}",k2,end=", ") 
    print()

In [ ]:
vals.get("pi3S", {})
# vals[temps_finis[t]+personnes_finies[p]]

# Génération tableau de candidates

In [ ]:
# Structure des clés / cases
personnes_finies = {"1SG":"1S", "2SG":"2S", "3SG":"3S", "1PL":"1P", "2PL":"2P", "3PL":"3P"}
temps_finis = {"présent":"pi", "imparfait":"ii", "passé":"ai", "futur":"fi", "subj. prés.":"ps", "subj. imparf.":"is", "conditionnel":"pc", "impératif":"pI"}

maxT={}
for t in temps_finis.values():
    maxT[t]=max([len(v) for k,v in vals.items() if t in k]) 
for t in ["inf","pP","ppMS","ppMP","ppFS","ppFP"]:
    maxT[t]=max([len(v) for k,v in vals.items() if t in k]) 

print(maxT)

In [ ]:
# On suppose que `vals` est structuré ainsi : vals[(temps, personne)] = {forme: pourcentage}

def textipa(text):
    return r"\textipan{%s}"%sampa2api(text)

def multirow(text,n):
    if n == 1:
        return text
    else:
        return r"\multirow{%d}{*}{%s}"%(n,text)

def escape_latex(text):
    """Échappe le signe % pour qu'il soit valide en LaTeX."""
    return str(text).replace('%', r'\%')

def format_cellule(dict_formes):
    """Formate le contenu d'une cellule avec retours à la ligne si multiple."""
    if not dict_formes:
        return "--"
    
    # Tri décroissant selon le pourcentage/poids
    items = sorted(dict_formes.items(), key=operator.itemgetter(0, 1), reverse=False)
    
    # Cas à 100% avec une seule forme (affiché sans pourcentage dans l'image si nécessaire, ou avec)
    # Dans l'image : si 100%, il est écrit "100% abwa" (ou juste en gras selon le cas).
    
    lines = []
    for forme, pct in items:
        if isinstance(pct, (int, float)):
            pct_str = f"{pct:.0%}".replace('%', r'\%')
        else:
            pct_str = str(pct)
        lines.append(f"{pct_str} {textipa(forme)}")
    
    # Si la cellule contient plusieurs formes, on utilise \shortstack pour les empiler
    if len(lines) > 1:
        return r"\newline ".join(lines)
    elif len(lines) == 1:
        return multirow(lines[0],maxT[tt])
    return "--"

In [ ]:
# --- GÉNÉRATION DU CODE LATEX ---

latex_code = []

latex_code.append(r"\begin{tabular}[t]{@{\hspace{.5ex}}l" + r"@{\hspace{.5ex}}p{13ex}" * len(personnes_finies) + r"@{\hspace{.5ex}}}")
latex_code.append(r"\toprule")
latex_code.append(r"\textbf{formes finies} & " + " & ".join([f"\\textsc{{{p.lower()}}}" for p in personnes_finies]) + r" \\")
latex_code.append(r"\midrule")

for t,tt in temps_finis.items():
    row=[multirow(t,maxT[tt])]
    for p,pp in personnes_finies.items():
        cell_data = vals.get(tt+pp, {})
        row.append(format_cellule(cell_data))
    
    
    
    # Lignes de séparation de sections comme dans l'image
    if t in ["futur", "subj. imparf.", "conditionnel"]:
        line_str = " & ".join(row)+r"\\"+"\n"+r"\midrule"
    else:
        if maxT[tt]>1:
            saut="[12pt]"
        else:
            saut="[3pt]"
        line_str = " & ".join(row) + r" \\"+saut
        
    latex_code.append(line_str)

latex_code.append(r"\bottomrule")
latex_code.append(r"\end{tabular}")

# ESPACEMENT ENTRE LES DEUX TABLEAUX
latex_code.append(r"\hfill")

# TABLEAU 2 : Formes non-finies (à droite)
latex_code.append(r"\begin{tabular}[t]{@{\hspace{.5ex}}l@{\hspace{.5ex}}p{1.6cm}@{\hspace{.5ex}}}")
latex_code.append(r"\toprule")
latex_code.append(r"\multicolumn{2}{c}{\textbf{formes non-finies}} \\")
latex_code.append(r"\midrule")

# Exemples de formes non-finies
formes_non_finies = [
    (multirow("infinitif",maxT["inf"]), vals.get("inf", {})),
    (multirow("part. prés.",maxT["pP"]), vals.get("pP", {})),
    (multirow(r"\textsc{m.sg}",maxT["ppMS"]), vals.get("ppMS", {})),
    (multirow(r"\textsc{m.pl}",maxT["ppMP"]), vals.get("ppMP", {})),
    (multirow(r"\textsc{f.sg}",maxT["ppFS"]), vals.get("ppFS", {})),
    (multirow(r"\textsc{f.pl}",maxT["ppFP"]), vals.get("ppFP", {})),
]

for label, cell_data in formes_non_finies:
    cell_formatted = format_cellule(cell_data)
    latex_code.append(f"{label} & {cell_formatted} \\\\")
    if "infinitif" in label:
        latex_code.append(r"\midrule")
    elif "part. prés." in label:
        latex_code.append(r"\midrule")
        latex_code.append(r"\multicolumn{2}{c}{part. passé} \\")

latex_code.append(r"\bottomrule")
latex_code.append(r"\end{tabular}")

# Affichage du résultat
print("\n".join(latex_code))
pyperclip.copy("\n".join(latex_code))

# Faire le graphe des cliques

In [ ]:
transformations=nx.DiGraph()
paires={}
for c in ordStemCells:
    # print(c,vals[c])
    for cc in ordStemCells:
        for k in vals[c]:
            kDist=regles[(c,cc)].sortirForme(k,contextFree=False)
            commonDist=kDist.keys() & vals[cc].keys()
            for kk in commonDist:
                transformations.add_edge(c+"-"+k,cc+"-"+kk, weight=kDist[kk])

In [ ]:

def to_undirected_mean_weight(G: nx.DiGraph, weight_key="weight"):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))

    for u, v, data in G.edges(data=True):
        a, b = sorted((u, v))  # paire non orientée

        if H.has_edge(a, b):
            # on met à jour la moyenne en mode somme/nb
            H[a][b]["_sum"] += data.get(weight_key, 1)
            H[a][b]["_n"] += 1
        else:
            H.add_edge(a, b,
                       weight=data.get(weight_key, 1),
                       _sum=data.get(weight_key, 1),
                       _n=1)

    # finaliser la moyenne
    for a, b in list(H.edges()):
        H[a][b]["weight"] = H[a][b]["_sum"] / H[a][b]["_n"]
        del H[a][b]["_sum"]
        del H[a][b]["_n"]

    return H

relations = to_undirected_mean_weight(transformations)

cliques = list(nx.find_cliques(relations))
for clique in cliques:
    print(len(clique),", ".join(sorted(clique)))
    pClique=[]
    for node1 in clique:
        for node2 in clique:
            pRelation=relations[node1][node2]["weight"]
            pClique.append(pRelation)
            # print(node1,node2,pRelation)
    print(np.mean(pClique))

In [ ]:
for clique in cliques:
    print(len(clique),", ".join(sorted(clique)))
    # nodes1=[n for n in clique if "pi3S" in n or "fi3S" in n]
    pClique=[]
    for node1 in clique:
        for node2 in clique:
            pRelation=relations[node1][node2]["weight"]
            pClique.append(pRelation)
            # print(node1,node2,pRelation)
    print(np.mean(pClique))

In [ ]:
# (set(cliques[0]) & 
(set(cliques[0]) - set(cliques[1]))

In [ ]:
subgraph = relations.subgraph(cliques[0])
plt.figure(figsize=(50, 50))
pos = nx.circular_layout(subgraph)
nx.draw_networkx(
        subgraph,
        pos=pos,
        node_color="violet",
        node_size=50,
        font_size=10,
        edge_color="orange",
        width=2,
    )
# plt.savefig("TEMP.png",dpi=300,bbox_inches="tight")

In [ ]:
regles[("inf","ai2S")].sortirForme("abwar",contextFree=False)

In [ ]:
import itertools
import matplotlib.pyplot as plt
import networkx as nx

# Definition des trois listes de nœuds
liste1 = cliques[0]

liste2 = cliques[1]

liste3 = cliques[2]

listes = [liste1, liste2, liste3]

# Création du graphe non-orienté
G = nx.Graph()

# Ajout des arêtes : pour chaque liste, on connecte tous les nœuds deux à deux
for liste in listes:
    # Nettoyage des espaces insécables éventuels dans les chaînes
    noeuds_nettoyes = [noeud.strip() for noeud in liste]
    G.add_edges_from(itertools.combinations(noeuds_nettoyes, 2))

# Visualisation du graphe
plt.figure(figsize=(14, 10))

# Disposition des nœuds (spring layout donne un bon rendu pour les cliques)
pos = nx.spring_layout(G, k=0.5, seed=42)

# Dessin des nœuds, des arêtes et des étiquettes
nx.draw_networkx_nodes(G, pos, node_size=1500, node_color="skyblue", alpha=0.9)
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5, edge_color="gray")
nx.draw_networkx_labels(G, pos, font_size=8, font_family="sans-serif")

plt.title("Graphe non-orienté interconnecté", fontsize=14)
plt.axis("off")
plt.tight_layout()

# Affichage
plt.show()

# nx.nx_pydot.write_dot(G, "mon_graphe.dot")

In [ ]:
for k,v in regles[("inf","pP")].patrons.items():
    print(k,v)
    m=re.split("(\[[^\]]*\])",v)
    print(k,end=" : ")
    for e in m:
        if e.startswith("[") and e.endswith("]"):
            print(fs.lattice[e[1:-1]].intent,end="")
        elif e!="":
            print(e,end="")
    print()
    print()

In [ ]:
regles[("ai3S","ii3S")].sortirForme("bry",False)


# Extraction des règles

### préparatifs extraction

In [ ]:
def diff(mot1,mot2):
    result=[]
    diff1=""
    diff2=""
    same=""
    vide="."
    lmax=max(len(mot1),len(mot2))
    lmin=min(len(mot1),len(mot2))
    for index in range(lmax):
        if index < lmin:
            if mot1[index]!=mot2[index]:
                diff1+=mot1[index]
                diff2+=mot2[index]
                same+=vide
            else:
                same+=mot1[index]
                diff1+=vide
                diff2+=vide
        elif index < len(mot1):
            diff1+=mot1[index]
        elif index < len(mot2):
            diff2+=mot2[index]
    diff1=diff1.lstrip(".")
    diff2=diff2.lstrip(".")
#    return (same,diff1,diff2,diff1+"_"+diff2)
    return (diff1+"-"+diff2)

In [ ]:
def patron2regexp(morceaux):
    result="^"
    for morceau in morceaux:
        if morceau=="*":
            result+="(.*)"
        elif len(morceau)>1:
            result+="(["+morceau+"])"
        else:
            result+=morceau
    result+="$"
    result=result.replace(")(","")
    return result

In [ ]:
class formesPatron:
    '''
    Accumulateur de formes correspondant à un patron pour calcul de la Généralisation Minimale (cf. MGL)
    '''
    def __init__(self):
        self.formes=[]

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterForme(self,forme):
        self.formes.append(forme)
        
    def calculerGM(self):
        minLongueur=len(min(self.formes, key=len))
        maxLongueur=len(max(self.formes, key=len))
        if debug: 
            print (minLongueur, maxLongueur, file=logfile)
            print (minLongueur, maxLongueur)
        positions=[]
        if maxLongueur>minLongueur:
            positions.append("*")
        for i in range(minLongueur, 0, -1):
            phonemes=set([x[-i] for x in self.formes])
            # print("phonemes",phonemes)
            if debug: 
                print (phonemes, file=logfile)
                print (phonemes)
            if "." in phonemes:
                positions.append(".")
            else:
                positions.append("".join(fs.lattice[phonemes].extent))
        return patron2regexp(positions)

class pairePatrons:
    '''
    Accumulateur de triplets (f1,f2,patron) correspondant à une paire pour calcul des Généralisations Minimales (cf. MGL)
    '''
    def __init__(self,case1,case2):
        self.patrons1={}
        self.patrons2={}
        self.case1=case1
        self.case2=case2

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterFormes(self,forme1,forme2,patron):
#        print (forme1,forme2,patron, file=logfile)
        patron12=patron
        (pat1,pat2)=patron.split("-")
        patron21=pat2+"-"+pat1
#        print (patron12,patron21, file=logfile)
        if not patron12 in self.patrons1:
            self.patrons1[patron12]=formesPatron()
        self.patrons1[patron12].ajouterForme(forme1)
        if not patron21 in self.patrons2:
            self.patrons2[patron21]=formesPatron()
        self.patrons2[patron21].ajouterForme(forme2)
        
        
    def calculerGM(self):
        resultat1={}
        for patron in self.patrons1:
            if debug: 
                print ("patron1", patron, file=logfile)
                print ("patron1", patron)
            resultat1[patron]=self.patrons1[patron].calculerGM()
        resultat2={}
        for patron in self.patrons2:
            if debug: 
                print ("patron2", patron, file=logfile)
                print ("patron2", patron)
            resultat2[patron]=self.patrons2[patron].calculerGM()
        return (resultat1,resultat2) 

### Extractions des règles

In [ ]:
from itertools import combinations

In [ ]:
def feat2phonrule(text):
    feat=fs.lattice[text[1:-1]].intent
    return r"\phonfeat[l]{"+r"\\".join(feat)+"}"

def translateRule(motif,context):
    result=[]
    m=[chunk for chunk in re.split("(\[[^\]]*\])",context) if chunk!=""]
    for e in m:
        if e.startswith("[") and e.endswith("]"):
            # result.append(fs.lattice[e[1:-1]].intent)
            result.append(feat2phonrule(e))
        elif e!="" and e!=None:
            result.append(e)
    # print(motif,context," ".join(result))
    return " ".join(result)
    
def translateRules(regles1,MGL=False):
    results=[]
    for motif,context in regles1.items():
        results.append(translateRule(motif,context))

    pattern = r"(\\phonfeat(?:\[[^\]]*\])?\{[^}]*\}(?:(?!\\phonfeat).)*)$"
    match = re.search(pattern, " ".join(results))

    if MGL and match:
        # print(match.group(0))
        return match.group(0)
    else:
        return " ".join(results)

In [ ]:
print('r"'+translateRule("E-â","[aE][mnl]E")+'"')
# feat2phonrule("[lk]")

In [ ]:
case1,case2="inf","pP"


tPaires=[
    ("kalE","kalâ"),
    ("kalmE","kalmâ"),
    ("pasE","pasâ"),
    ("parlE","parlâ"),
    ("pâsE","pâsâ"),
    ("truvE","truvâ"),
    ("lEsE","lEsâ"),
    ("arivE","arivâ"),
    ("dOnE","dOnâ"),
    ("r6gardE","r6gardâ"),
    ("rEstE","rEstâ"),
    ("arEtE","arEtâ"),
    ("tyHE","tyHâ"),
    ("d6mâdE","d6mâdâ"),
    ("SErSE","SErSâ"),
    ("sEdE","sEdâ"),
       ]

paires=tPaires[:]

lCombinaisons=[]
for r in range(len(paires)+1):
    lCombinaisons.extend(combinations(paires,r))

sRegles=set()
for c in lCombinaisons:
    # print(c)
    patrons=pairePatrons(case1,case2)
    classes=paireClasses(case1,case2)
    for (f1,f2) in c:
        # print(f1,f2)
        patrons.ajouterFormes(f1,f2,diff(f1,f2))
    (regles1,regles2)=patrons.calculerGM()
    # print(regles1)
    sRegles.add(translateRules(regles1,MGL=False))
len(paires),len(lCombinaisons),len(sRegles)

In [ ]:
patrons=pairePatrons(case1,case2)
classes=paireClasses(case1,case2)
for (f1,f2) in tPaires:
    patrons.ajouterFormes(f1,f2,diff(f1,f2))
(regles1,regles2)=patrons.calculerGM()
print(regles1)
print(translateRules(regles1,MGL=False))

In [ ]:
sRegles

In [ ]:
for i, (f1,f2) in enumerate(paires):
    print(", ".join([c1+"-"+c2 for (c1,c2) in paires[:i+1]]))
    patrons.ajouterFormes(f1,f2,diff(f1,f2))
    (regles1,regles2)=patrons.calculerGM()
    translateRule(regles1)
    print()
    print()

In [ ]:
p=("ii3S","ppMS")
k="wajE-y"
v=regles[p].patrons[k]

In [ ]:
print(k,v)
m=re.split("(\[[^\]]*\])",v)
print(k,end=" : ")
for e in m:
    if e.startswith("[") and e.endswith("]"):
        print(fs.lattice[e[1:-1]].intent,end="")
    elif e!="":
        print(e,end="")

In [ ]:
regles[p].sortirForme("brwajE")

In [ ]:
from scipy.stats import entropy

entropy([1,1],base=2)

# TEMP

In [49]:
regles[("ii3S","ppMS")].patrons

{'-': '^(.*[ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwHE96Oêûô])E$',
 'sE-': '^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô][iEe])sE$',
 'E-y': '^(.*[ptkbdgfsSvzZmnJNjrwHE96Oêûô][jrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNlr])E$',
 'OlvE-u': '^([bdEa][pbjiEe])sOlvE$',
 'jE-': '^(.*[ptbdfsvz][jrwH][iEe])jE$',
 'wasE-y': '^(.*)krwasE$',
 'E-i': '^(.*[ptkbdgfsSvzZmnJNjrwHE96aOêûâô][jrwHiyEe926auOoêûâô][ptbdfsvzmnJj])E$',
 'ErE-i': '^(.*[E96aOêûâô])kErE$',
 'aJE-ê': '^(.*)waJE$',
 'EtE-i': '^(.*)mEtE$',
 '9vE-y': '^(.*[tdszl])9vE$',
 'EsE-y': '^(.*[pkbgfvr][E96aO][ptbdfsvzmnr])EsE$',
 '9nE-i': '^(.*)pr9nE$',
 'E-yHE': '^argE$',
 'yHE-E': '^argyHE$',
 'EjE-i': '^(.*)sEjE$',
 'wajE-i': '^(.*[rE6a])swajE$',
 'EJE-ê': '^(.*[ptkbdgfsSvzZlr])EJE$',
 'avE-y': '^(.*)avE$',
 'vE-': '^(.*[bvr][iy])vE$',
 'zE-': '^(.*[ptkbdgfsSvzZjlrwH][iyEe926uOo])zE$',
 'EzE-y': '^(.*[tdl])EzE$',
 'HE-': '^(.*[fsvzrE96Oêûô])klyHE$',
 '9zE-E': '^(.*)f9zE$',
 'rE-Er': '^(.*[uOo][fv])rE$',
 'wajE-y': '^(.*[vr])wajE$',

# Règles S4

In [10]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/S4/"
rulesFile="vlexique2-S4-Regles.pkl"
fS4Rules=repFiles+rulesFile
with open(fS4Rules, 'rb') as input:
    reglesS4 = pickle.load(input)

In [31]:
reglesS4[("ai1S","is2S")].patrons

{'-s': '^(.*[bdEa])i$'}

In [18]:
for case in ordStemCells:
    print(case)
    pCase=reglesS4[(case,"is3S")].patrons
    for k,v in pCase.items():
        if k.endswith('-') or k.endswith("E") :
            print(k,v)
    print()

pi1S

pi2S

pi3S

pi1P

pi2P

pi3P

ii1S

ii2S

ii3S

ii1P

ii2P

ii3P

fi1S

fi2S

fi3S

fi1P

fi2P

fi3P

pc1S

pc2S

pc3S

pc1P

pc2P

pc3P

ps1S

ps2S

ps3S

ps1P

ps2P

ps3P

ai1S
- ^(.*[iyEe92êû])$

ai2S
- ^(.*[iyEe92a])$

ai3S
- ^(.*[iyEe92aêû])$

ai1P
m- ^(.*[iyEe92aêû])m$

ai2P
t- ^f([iy])t$

ai3P
r- ^(.*[iyEe92êû])r$

is1S
s- ^(.*)ys$

is2S

is3S
- ^(.*[iyEe92aêû])$

is1P
sjô- ^fysjô$

is2P
sjE- ^(.*)ysjE$

is3P
s- ^(.*)ys$

pI2S

pI1P

pI2P

inf

pP

ppMS
- ^(.*[iy])$

ppMP
- ^(.*[iy])$

ppFS
- ^(.*)y$
z- ^priz$

ppFP
- ^(.*)y$
z- ^priz$



In [23]:
for case in ordStemCells:
    print(case)
    pCase=reglesS4[(case,"ai3S")].patrons
    for k,v in pCase.items():
        if k=='-' or k.endswith("E") :
            print(k,v)
    print()

pi1S
- ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])i$

pi2S
- ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])i$

pi3S
- ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])i$

pi1P

pi2P

pi3P
- ^(.*[jrwH])i$

ii1S

ii2S

ii3S

ii1P

ii2P

ii3P

fi1S

fi2S

fi3S

fi1P

fi2P

fi3P

pc1S

pc2S

pc3S

pc1P

pc2P

pc3P

ps1S
- ^(.*[jrwH])i$

ps2S
- ^(.*[jrwH])i$

ps3S
- ^(.*[jrwH])i$

ps1P

ps2P

ps3P
- ^(.*[jrwH])i$

ai1S
- ^(.*[iyEe92êû])$

ai2S
- ^(.*[iyEe92aêû])$

ai3S
- ^(.*[iyEe92aêû])$

ai1P

ai2P

ai3P

is1S

is2S

is3S
- ^(.*[iyEe92aêû])$

is1P

is2P

is3P

pI2S
- ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])i$

pI1P

pI2P

inf

pP

ppMS
- ^(.*[iy])$

ppMP
- ^(.*[iy])$

ppFS
- ^(.*[iy])$

ppFP
- ^(.*[iy])$



In [29]:
erreurs=["accepter", "adorer", "aimer", "amener", "attirer", "avérer", "brûler", "charger", "commencer", "contrôler", "couler", "créer", "demander", "discuter", "durer", "embrasser", "emporter", "engager", "enterrer", "entrer", "envoler", "essayer", "forger", "gagner", "glisser", "ignorer", "imposer", "jeter", "juger", "laisser", "lever", "livrer", "manger", "marquer", "montrer", "obliger", "oser", "parler", "payer", "placer", "porter", "poser", "provoquer", "raconter", "rappeler", "rapporter", "refuser", "regarder", "rencontrer", "renoncer", "retirer", "rouler", "régner", "réveiller", "révéler", "sacrifier", "sauver", "sembler", "sonner", "tenter", "tomber", "toucher", "tourner", "tuer", "éloigner", "épouser", "éveiller", ]

ai1S=["abandonner", "accepter", "accompagner", "acheter", "adorer", "aider", "aimer", "aller", "appeler", "apporter", "arracher", "arriver", "arrêter", "assurer", "cacher", "cesser", "changer", "chercher", "commencer", "compter", "continuer", "crier", "croiser", "demander", "diriger", "donner", "décider", "embrasser", "emmener", "engager", "enlever", "entrer", "envoyer", "espérer", "essayer", "expliquer", "fermer", "frapper", "gagner", "garder", "glisser", "ignorer", "imaginer", "installer", "interroger", "jeter", "jouer", "jurer", "laisser", "lancer", "lever", "manger", "marcher", "monter", "montrer", "observer", "oser", "oublier", "parler", "passer", "penser", "pleurer", "plonger", "porter", "poser", "pousser", "prier", "proposer", "précipiter", "préférer", "préparer", "présenter", "quitter", "raconter", "ramener", "rappeler", "refuser", "regarder", "remarquer", "rencontrer", "rentrer", "rester", "retirer", "retourner", "retrouver", "réaliser", "récupérer", "réveiller", "rêver", "sauter", "serrer", "souhaiter", "tenter", "tomber", "tourner", "travailler", "traverser", "trouver", "tuer", "utiliser", "vérifier", "écouter", "épouser", "éprouver", "éveiller"]

ai3P=["abandonner", "accepter", "accorder", "accuser", "acheter", "adopter", "adorer", "affirmer", "affluer", "affronter", "aider", "aimer", "aller", "allumer", "amener", "annoncer", "appeler", "apporter", "approcher", "arracher", "arranger", "arriver", "arrêter", "assister", "assurer", "attacher", "attaquer", "attirer", "attraper", "augmenter", "avancer", "avérer", "balayer", "baptiser", "blesser", "briser", "brûler", "cacher", "capturer", "causer", "cesser", "changer", "chanter", "charger", "chasser", "chercher", "commencer", "continuer", "contrôler", "couler", "couper", "creuser", "crier", "croiser", "créer", "céder", "danser", "demander", "diriger", "discuter", "disperser", "disputer", "diviser", "divorcer", "donner", "dresser", "durer", "débarquer", "débuter", "déchirer", "décider", "déclarer", "déménager", "déposer", "dérouler", "détourner", "développer", "effondrer", "embarquer", "embrasser", "emmener", "emménager", "emparer", "emporter", "empêcher", "enchaîner", "enfermer", "engager", "enlever", "entamer", "enterrer", "entourer", "entraîner", "entrer", "envelopper", "envoler", "envoyer", "essayer", "estimer", "exiger", "expliquer", "exploser", "fermer", "foncer", "fonder", "forcer", "forger", "former", "frapper", "gagner", "garder", "glisser", "hurler", "imposer", "inspirer", "installer", "inventer", "inviter", "jeter", "jouer", "juger", "jurer", "laisser", "lancer", "lever", "libérer", "livrer", "lutter", "lâcher", "manger", "marcher", "marier", "marquer", "massacrer", "mener", "monter", "montrer", "moquer", "multiplier", "nommer", "obliger", "occuper", "opposer", "ordonner", "organiser", "oser", "oublier", "parler", "partager", "participer", "passer", "payer", "penser", "piller", "placer", "pleurer", "plonger", "porter", "poser", "pousser", "profiter", "proposer", "provoquer", "précipiter", "précéder", "préparer", "présenter", "prêter", "pénétrer", "quitter", "raconter", "ramener", "rapporter", "rassembler", "rattraper", "rebeller", "recruter", "refuser", "regarder", "regrouper", "remarquer", "remonter", "rencontrer", "renoncer", "rentrer", "renverser", "renvoyer", "repousser", "rester", "retirer", "retourner", "retrouver", "rouler", "ruer", "réaliser", "réfugier", "régner", "résister", "réveiller", "révolter", "révéler", "sacrifier", "sauter", "sauver", "sembler", "semer", "signer", "sombrer", "sonner", "soulever", "succéder", "séparer", "tarder", "tenter", "terminer", "tirer", "tomber", "toucher", "tourner", "transformer", "travailler", "traverser", "traîner", "trouver", "tuer", "utiliser", "voler", "voter", "voyager", "écarter", "échanger", "échapper", "échouer", "éclater", "écouler", "écouter", "écraser", "écrier", "écrouler", "élever", "éloigner", "émerger", "étudier"]

set(erreurs)-set(ai3P), set(erreurs)-set(ai1S)

({'ignorer', 'rappeler', 'épouser', 'éveiller'},
 {'amener',
  'attirer',
  'avérer',
  'brûler',
  'charger',
  'contrôler',
  'couler',
  'créer',
  'discuter',
  'durer',
  'emporter',
  'enterrer',
  'envoler',
  'forger',
  'imposer',
  'juger',
  'livrer',
  'marquer',
  'obliger',
  'payer',
  'placer',
  'provoquer',
  'rapporter',
  'renoncer',
  'rouler',
  'régner',
  'révéler',
  'sacrifier',
  'sauver',
  'sembler',
  'sonner',
  'toucher',
  'éloigner'})